### ЗАДАЧА: Реестр пропусков коворкинга

Администратор коворкинга ведёт реестр пропусков резидентов.
Нужно собрать модель, которая позволит:
- загрузить пропуска в единый реестр,
- посмотреть только активные пропуска,
- отфильтровать резидентов по тарифу,
- посчитать суммарный остаток дней,
- понять, как меняется реестр после паузы, списания дня и продления.

В данных есть разные тарифы и статусы,
поэтому важно корректно валидировать plan, status и изменение состояния объекта.
        


In [ ]:
rows = [
    'PS-100|Alice|flex|12|active',
    'PS-101|Bob|fixed|20|paused',
    'PS-102|Team Rocket|team|0|expired',
    'PS-103|Diana|flex|6|active',
]

class CoworkingPass:
    allowed_plans = {'flex', 'fixed', 'team'}
    allowed_statuses = {'active', 'paused', 'expired'}

    def __init__(self, pass_id, member_name, plan, days_left, status):
        # TODO: проверить plan и status, иначе raise ValueError(...)
        if plan not in self.allowed_plans:
            raise ValueError("нет такого плана")
        if status not in self.allowed_statuses:
            raise ValueError("нет такого статуса")
        # TODO: сохранить pass_id, member_name, plan, status
        self.pass_id = pass_id
        self.member_name = member_name
        self.plan = plan
        self.status = status
        # TODO: days_left хранить через внутреннее поле self._days_left
        self._days_left = None
        # TODO: значение days_left пропустить через property/setter
        self.days_left = days_left

    @property
    def days_left(self):
        # TODO: вернуть текущее число оставшихся дней
        return self._days_left

    @days_left.setter
    def days_left(self, value):
        # TODO: привести value к int
        value = int(value)
        # TODO: если value < 0 -> raise ValueError('Days must be >= 0')
        if value < 0:
            raise ValueError('Days must be >= 0')
        # TODO: сохранить результат в self._days_left
        self._days_left = value

    def use_day(self):
        # TODO: если статус не 'active' -> raise ValueError(...)
        if self.status != 'active':
            raise ValueError("некорректный статус")
        # TODO: если days_left == 0 -> raise ValueError(...)
        if self.days_left == 0:
            raise ValueError("не может быть нулем")
        # TODO: уменьшить days_left на 1
        self.days_left -= 1
        # TODO: если после списания days_left == 0, перевести статус в 'expired'
        if self.days_left == 0:
            self.status = 'expired'

    def pause(self):
        # TODO: если статус 'expired' -> raise ValueError(...)
        if self.status == 'expired':
            raise ValueError("истекший статус")
        # TODO: перевести пропуск в 'paused'
        self.status = 'paused'

    def resume(self):
        # TODO: если days_left == 0 -> raise ValueError(...)
        if self.days_left == 0:
            raise ValueError("дней не осталось")
        # TODO: перевести пропуск в 'active'
        self.status = 'active' 

    def renew(self, extra_days):
        # TODO: привести extra_days к int
        extra_days = int(extra_days)
        # TODO: если extra_days <= 0 -> raise ValueError(...)
        if extra_days <= 0:
            raise ValueError("число дней не может быть отрицательным")
        # TODO: увеличить days_left
        self.days_left += extra_days
        # TODO: если days_left > 0 и статус был 'expired', перевести в 'active'
        if self.days_left > 0 and self.status == 'expired':
            self.status = 'active'

    @classmethod
    def from_row(cls, row):
        # TODO: split по '|'
        parts = row.split('|')
        # Проверка количества частей должна идти до распаковки
        if len(parts) != 5:
            raise ValueError("Неверный формат записи")
        # TODO: ожидать 5 частей: pass_id, member_name, plan, days_left, status
        pass_id, member_name, plan, days_left, status = parts
        # TODO: вернуть CoworkingPass(...)
        return cls(pass_id, member_name, plan, days_left, status)

    def __repr__(self):
        # TODO: вернуть строку вида CoworkingPass(pass_id='...', member_name='...', status='...', days_left=...)
        return f"CoworkingPass(pass_id='{self.pass_id}', member_name='{self.member_name}', status='{self.status}', days_left={self.days_left})"


class CoworkingRegistry:
    def __init__(self):
        self.items = []

    def add(self, coworking_pass):
        # TODO: добавить объект в self.items
        self.items.append(coworking_pass)

    def load(self, rows):
        # TODO: для каждой строки создать CoworkingPass.from_row(row)
        for row in rows:
            coworking_pass = CoworkingPass.from_row(row)
            # TODO: добавить объект в реестр через add(...)
            self.add(coworking_pass)

    def active_passes(self):
        # TODO: вернуть список пропусков со статусом 'active'
        return [p for p in self.items if p.status == 'active']

    def by_plan(self, plan):
        # TODO: вернуть список пропусков нужного тарифа
        return [p for p in self.items if p.plan == plan]

    def total_days_left(self):
        # TODO: вернуть суммарное число оставшихся дней
        return sum(p.days_left for p in self.items)

    def status_summary(self):
        # TODO: собрать dict вида status -> count
        summary = {}
        for p in self.items:
            summary[p.status] = summary.get(p.status, 0) + 1
        return summary

    def find(self, pass_id):
        # TODO: вернуть пропуск по pass_id или None
        for p in self.items:
            if p.pass_id == pass_id:
                return p
        return None

    def largest_balance(self):
        # TODO: найти пропуск с максимальным days_left
        if not self.items:
            return None
        max_pass = max(self.items, key=lambda p: p.days_left)
        # TODO: вернуть tuple(pass_id, days_left)
        return (max_pass.pass_id, max_pass.days_left)

registry = CoworkingRegistry()

# TODO: загрузить rows в registry
registry.load(rows)

# TODO: вывести все пропуска
print("Все пропуска:")
for pass_obj in registry.items:
    print(pass_obj)

# TODO: вывести active_passes()
print("\nАктивные пропуска:")
for pass_obj in registry.active_passes():
    print(pass_obj)

# TODO: вывести by_plan('flex')
print("\nПропуска тарифа 'flex':")
for pass_obj in registry.by_plan('flex'):
    print(pass_obj)

# TODO: вывести total_days_left()
print(f"\nСуммарное число оставшихся дней: {registry.total_days_left()}")

# TODO: вывести status_summary()
print(f"Сводка по статусам: {registry.status_summary()}")

# TODO: найти пропуск 'PS-101', возобновить его и вывести status_summary()
pass_101 = registry.find('PS-101')
if pass_101:
    pass_101.resume()
    print(f"\nПосле возобновления пропуска PS-101 сводка по статусам: {registry.status_summary()}")

# TODO: найти пропуск 'PS-100', списать один день и вывести объект
pass_100 = registry.find('PS-100')
if pass_100:
    pass_100.use_day()
    print(f"\nПосле списания дня пропуск PS-100: {pass_100}")


# TODO: найти пропуск 'PS-102', продлить на 5 дней и вывести объект
pass_102 = registry.find('PS-102')
if pass_102:
    pass_102.renew(5)
    print(f"\nПосле продления на 5 дней пропуск PS-102: {pass_102}")

# TODO: вывести largest_balance()
largest = registry.largest_balance()
if largest:
    print(f"\nПропуск с максимальным балансом: pass_id='{largest[0]}', days_left={largest[1]}")
else:
    print("\nРеестр пуст, нет пропусков для анализа баланса")